# Session 2 · Day 1 - Administrative Boundaries, CRS, and the PSGC Hierarchy

Every TESDA question about *where* resolves to an administrative unit: how many assessment centers sit in Caloocan, which municipalities have none, whether demand from Rizal is being served by centers in Marikina and Pasig. Answering those needs the **shapes** of cities and municipalities, not only their names. This notebook loads the real published barangay-boundary file straight from the public bucket, reads its layers, geometry type, and coordinate system, decodes the Philippine Standard Geographic Code (PSGC) hierarchy from region down to barangay, dissolves barangays up to their city or municipality, and finds the neighbors of the National Capital Region with a spatial query. *Modeling is Session 3.*

### What this notebook covers
1. Inspect the boundary file (layers, geometry type, coordinate system) before loading every polygon
2. Stream one map window over the network, and read the boundary table one row at a time
3. The coordinate-system lesson: why land area measured in square degrees is meaningless, and which system to measure in
4. Decode the PSGC hierarchy (adm1 to adm4) and filter the National Capital Region by code
5. Dissolve barangays up to the city or municipality, the spatial version of a group-by
6. Find a region's neighbors with a spatial adjacency query, and measure the shared border
7. Build a reusable study-area polygon for later notebooks

> **How to use this notebook**
> - ▶ **Run each cell in order** (`Shift+Enter`). Later cells depend on earlier ones.
> - ✏️ **Exercise cells** have `### START CODE HERE ###` … `### END CODE HERE ###`. Fill in code between those lines only.
> - ✅ After an exercise, run the **check cell**. It recomputes the reference and reports a pass or a fix to make.
> - 🐞 **Spot-the-flaw cells** show a tempting shortcut that gives a wrong answer. Read the gap before moving on.
> - 🎛️ **Play-around cells** are open to tweak: change a value, re-run, compare.
>
> _Real boundary data loads over the network from a public bucket. No Google Drive mount and no local paths are needed._

In [ ]:
# Run once per session (skip if already installed).
!pip -q install geopandas shapely pyproj folium mapclassify matplotlib

In [ ]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import matplotlib.pyplot as plt

BASE = "https://storage.googleapis.com/tesda-datasets/data/"
# A GeoPackage read through GDAL's /vsicurl/ driver: the file stays on the bucket and
# only the bytes a query needs travel over the network.
BOUNDARIES = "/vsicurl/" + BASE + "consolidated_geodata_matched.gpkg"
LAYER = "consolidated_geodata_matched"

STORE_CRS = 4326     # WGS 84, degrees of longitude and latitude, the storage and web-map system
METRIC_CRS = 32651   # UTM zone 51N, meters, for measuring length and area across Luzon

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## Inspecting the file before loading it

Boundary data usually arrives in one of two formats, and the difference is worth knowing before anything is read:

- A **shapefile** is really a *set* of files that must travel together: `.shp` (the shapes), `.dbf` (the attribute table), `.shx` (an index), and `.prj` (the coordinate system). Lose the `.prj` and the numbers in the file no longer have a known meaning. Shapefiles also cut column names to 10 characters and hold one layer each, so a full boundary set comes as a folder with one shapefile per level.
- A **GeoPackage** (`.gpkg`) is a single file, a small SQLite database, that holds one or more named layers with full-length column names and the coordinate system built in.

The file here is a GeoPackage. Its header can be read without loading a single polygon, which is the right first move for a file with tens of thousands of shapes.

In [ ]:
info = pyogrio.read_info(BOUNDARIES, layer=LAYER)
print("layer name:    ", info["layer_name"])
print("geometry type: ", info["geometry_type"])
print("features:      ", f"{info['features']:,}")
print("coordinate sys:", info["crs"])
print("columns:       ", ", ".join(info["fields"][:12]), "...")

The header reports roughly 42,000 **MultiPolygon** features in **EPSG:4326**. That feature count is the clue to the grain: the Philippines has about 1,640 cities and municipalities but about 42,000 barangays, so this file is at the **barangay** level. One row is one barangay polygon, and a city will have to be built by merging its barangays later.

## Streaming one map window

Loading 42,000 polygons over the network to study one metropolitan area would be wasteful. A **bounding box** passed to `read_file` tells GDAL to return only the barangays whose extent overlaps that rectangle, so the read stays small and fast. The box below covers the National Capital Region and a ring of surrounding territory, in longitude and latitude order `(min_lon, min_lat, max_lon, max_lat)`.

In [ ]:
BBOX = (120.85, 14.30, 121.25, 14.82)   # (min_lon, min_lat, max_lon, max_lat) around Metro Manila

gdf = gpd.read_file(BOUNDARIES, bbox=BBOX)
gdf["geometry"] = gdf.geometry.make_valid()   # repair any self-touching rings before measuring
print(f"{len(gdf):,} barangay polygons in the window")
print("coordinate system:", gdf.crs)
print("geometry types:   ", gdf.geom_type.value_counts().to_dict())

## Boundaries are tables too

Just as roads and centers were rows in earlier notebooks, each **boundary is one row**: a set of code and name columns, then a `geometry` column holding the polygon. The PSGC columns come in two spellings of the same hierarchy: `adm1_pcode` through `adm4_pcode` are the humanitarian **p-codes** (`PH13`, `PH1307501`, …), and `adm1_en` through `adm3_en` are the names. The p-code is the reliable key, because names are spelled inconsistently across datasets while codes are not.

In [ ]:
name_cols = ["adm1_pcode", "adm2_pcode", "adm3_pcode", "adm4_pcode",
             "adm1_en", "adm2_en", "adm3_en", "adm4_en_psgc"]
gdf[name_cols].head(5)

## CRS: the most important setting in a map

A **Coordinate Reference System** (CRS) tells the computer what the numbers in `geometry` *mean*. The same place can be written in several systems, each with its own units:

| CRS | Units | Good for |
|---|---|---|
| **EPSG:4326** (WGS 84) | degrees of longitude and latitude | storing and sharing data, GPS, web maps |
| **EPSG:32651** (UTM zone 51N) | meters | measuring distance, area, and buffers across most of the Philippines |
| EPSG:3857 (Web Mercator) | distorted "meters" | drawing web-map tiles only |

Mixing systems is the most common silent error in geospatial work: nothing crashes, and the result is simply wrong. The clearest example is land area. Area computed while the data is still in degrees comes out in "square degrees", a unit with no fixed size on the ground, because a degree of longitude covers fewer meters the farther a place sits from the equator.

In [ ]:
cities_deg = (gdf[gdf["adm1_pcode"] == "PH13"]
              .dissolve(by="adm3_pcode", as_index=False, aggfunc="first"))   # merge barangays per city
area_deg = cities_deg.geometry.area                      # EPSG:4326 -> square degrees (meaningless)
area_km2 = cities_deg.to_crs(METRIC_CRS).geometry.area / 1e6   # EPSG:32651 -> real square kilometers

pd.DataFrame({
    "adm3_pcode": cities_deg["adm3_pcode"],
    "area in square degrees": area_deg.round(5),
    "area in km2 (UTM 51N)": area_km2.round(1),
}).head(6)

The first column has no usable unit. **Rule of thumb: store in degrees (4326), measure in a meter-based system such as 32651, and check `.crs` whenever two datasets meet.** The next cell shows what goes wrong when that rule is ignored.

## 🐞 Spot the flaw - measuring area in degrees

A tempting shortcut: compute `.area` on the data as loaded (degrees), then convert to kilometers with the schoolbook figure "1 degree is about 111 km", so one square degree is `111.32**2` square kilometers. Run it against the true UTM area and read the error.

In [ ]:
naive_km2 = area_deg * (111.32 ** 2)        # pretend 1 degree = 111.32 km everywhere
error_pct = (naive_km2 - area_km2) / area_km2 * 100

flaw = pd.DataFrame({
    "naive (111 km/deg)": naive_km2.round(1),
    "true km2 (UTM 51N)": area_km2.round(1),
    "error %": error_pct.round(1),
}).head(6)
print(flaw.to_string(index=False))
print(f"\nmean absolute error: {error_pct.abs().mean():.1f}%")

Every naive value is about 4% too large, and in the same direction. The reason is the conversion itself: at roughly 14.6 degrees north, a degree of longitude spans about 107.6 km, not 111.32, so the longitude side of every shape is overstated. A 4% bias on every area quietly distorts any "centers per square kilometer" or demand-density map. Reprojecting to a meter-based CRS before measuring removes the bias entirely.

## The PSGC hierarchy: a code carries its own tree

A PSGC p-code is built level by level, so the code *contains* the hierarchy above it:

| Column | Level | Example | Count in the Philippines |
|---|---|---|---|
| `adm1_pcode` | Region | `PH13` | 18 |
| `adm2_pcode` | Province (or an NCR district) | `PH13075` | about 80 to 120 |
| `adm3_pcode` | **City / Municipality** | `PH1307501` | about 1,640 |
| `adm4_pcode` | Barangay | `PH1307501080` | about 42,000 |

Because the region prefix is inside every code, the National Capital Region is simply every row whose `adm1_pcode` is **`PH13`**, however the region name happens to be spelled. Filtering on the code avoids every name-spelling problem at once.

In [ ]:
ncr = gdf[gdf["adm1_pcode"] == "PH13"].copy()
print(f"NCR barangays in the window: {len(ncr):,}")
print(f"NCR cities/municipalities (distinct adm3_pcode): {ncr['adm3_pcode'].nunique()}")
print(f"adm2 label(s) carried for NCR: {sorted(ncr['adm2_en'].unique())}")
print(f"adm3 label(s) carried for NCR: {sorted(ncr['adm3_en'].unique())}")

Two quirks show up here, and both are normal for NCR. First, the region has no provinces: its `adm2` slot holds legislative **districts** instead, so every NCR barangay reports the same `adm2_en`. Second, this particular file leaves `adm3_en` as the placeholder "Metro Manila" for all 17 units, so the city names are not in the table and have to be supplied from the code. The mapping below keeps the 16 cities and the lone municipality, Pateros, keyed by `adm3_pcode`.

In [ ]:
NCR_NAMES = {
    "PH1303901": "City of Manila",      "PH1307401": "City of Mandaluyong",
    "PH1307402": "City of Marikina",    "PH1307403": "City of Pasig",
    "PH1307404": "Quezon City",         "PH1307405": "City of San Juan",
    "PH1307501": "City of Caloocan",    "PH1307502": "City of Malabon",
    "PH1307503": "City of Navotas",     "PH1307504": "City of Valenzuela",
    "PH1307601": "City of Las Pi\u00f1as",  "PH1307602": "City of Makati",
    "PH1307603": "City of Muntinlupa",  "PH1307604": "City of Para\u00f1aque",
    "PH1307605": "Pasay City",          "PH1307606": "Pateros",
    "PH1307607": "City of Taguig",
}
print(f"{len(NCR_NAMES)} NCR local government units named (16 cities + Pateros)")

## Dissolve: barangays up to cities

A **dissolve** merges every row that shares a key into one polygon. It is the spatial version of a group-by: group by `adm3_pcode`, and the barangays of each city fuse into one city shape. Grouping on the code rather than the (placeholder) name means the result does not depend on how the source file spelled anything.

In [ ]:
ncr_cities = ncr.to_crs(METRIC_CRS).dissolve(by="adm3_pcode", as_index=False, aggfunc="first")
ncr_cities["lgu"] = ncr_cities["adm3_pcode"].map(NCR_NAMES)
print(f"NCR local government units after dissolve: {len(ncr_cities)}  (expected 17)")
sorted(ncr_cities["lgu"])

**Domain check:** NCR has **16 cities plus the municipality of Pateros, 17 units in all**. A different count is a signal to look at the codes, where an unexpected `adm3_pcode` usually means a typo or an outdated PSGC version in the source.

## ✏️ Exercise - land area of each NCR city

The geometry in `ncr_cities` is already in `METRIC_CRS` (meters), so area is safe to measure. Add an `area_km2` column (square meters divided by `1e6`), then build `area_rank`: a pandas Series indexed by the city name `lgu`, holding the area in square kilometers, sorted from largest to smallest.

**Hint:** `ncr_cities.geometry.area` gives square meters; set the index with `.set_index("lgu")` and sort with `.sort_values(ascending=False)`.

In [ ]:
### START CODE HERE ###   (about 2 lines)
# 1) add ncr_cities["area_km2"] from the metric geometry
# 2) build area_rank: a Series indexed by "lgu", sorted largest-area first
area_rank = None
### END CODE HERE ###
print(area_rank)

## ✅ Check the work
Run this. It recomputes the reference and compares it to `area_rank`.

In [ ]:
try:
    ref = (ncr_cities.assign(_a=ncr_cities.geometry.area / 1e6)
           .set_index("lgu")["_a"].sort_values(ascending=False))
    assert area_rank is not None, "area_rank is None - fill in the code."
    assert isinstance(area_rank, pd.Series), f"expected a pandas Series, got {type(area_rank)}"
    assert set(area_rank.index) == set(ref.index), "some cities are missing"
    assert np.allclose(area_rank.sort_index().values, ref.sort_index().values), \
        "area values do not match - measure in the metric CRS and divide by 1e6"
    assert list(area_rank.index) == list(ref.index), "sort from largest area to smallest"
    print(f"\u2705 Passed! Largest: {area_rank.index[0]} ({area_rank.iloc[0]:.0f} km2); "
          f"smallest: {area_rank.index[-1]} ({area_rank.iloc[-1]:.1f} km2).")
except AssertionError as e:
    print("\u274c Not yet:", e)

## Finding a region's neighbors with a spatial query

Learners do not stop at city limits. Someone in San Mateo, Rizal may be closer to a center in Quezon City than to any center inside Rizal, so the study area has to include the territory bordering NCR. Typing that list of neighbors from memory is error-prone and does not transfer to other regions. A **spatial adjacency query** reads the answer off the geometry, and the same code works for any region.

There are two ways to ask "does this share a border", and the contrast is the lesson:

- **Strict `touches`:** the boundaries must meet exactly. Real boundary files carry tiny gaps and overlaps (*slivers*) between neighbors, so a strict test can miss a true neighbor whose edge is a few meters off.
- **Buffered `intersects`:** grow the NCR outline by a small tolerance, then keep whatever overlaps it. This is forgiving of slivers, but it can also catch a unit that only *almost* touches, for example across a river or a bay.

To separate real neighbors from near misses, the query also measures the **length of border shared** with NCR. A unit meeting NCR at a single corner shows only a sliver of shared border, so a minimum-length cut-off filters those out.

In [ ]:
gm = gdf.to_crs(METRIC_CRS)                       # everything in meters for buffering and length
ncr_outline = ncr_cities.union_all()               # 17 cities -> one NCR outline
BUFFER_M = 150                                      # tolerance for slivers
MIN_BORDER_M = 500                                  # keep a neighbor only above this shared length

zone = ncr_outline.buffer(BUFFER_M)
outside = gm[gm["adm1_pcode"] != "PH13"].copy()
near = outside.iloc[outside.sindex.query(zone, predicate="intersects")]      # barangays touching the zone
cand = outside[outside["adm3_pcode"].isin(near["adm3_pcode"].unique())].copy()   # and their whole units
neighbors = cand.dissolve(by="adm3_pcode", as_index=False, aggfunc="first")
neighbors["shared_border_km"] = neighbors.boundary.intersection(zone).length / 1000
neighbors["touches_ncr"] = neighbors.touches(ncr_outline)

print(f"strict touches finds {neighbors['touches_ncr'].sum()} neighbors; "
      f"buffered intersects finds {neighbors.intersects(zone).sum()}.")

In [ ]:
kept = neighbors[neighbors["shared_border_km"] * 1000 >= MIN_BORDER_M].copy()
table = (kept[["adm3_en", "adm2_en", "adm3_pcode", "shared_border_km", "touches_ncr"]]
         .sort_values("shared_border_km", ascending=False)
         .reset_index(drop=True))
print(f"{len(kept)} adjacent units kept, in {kept['adm2_en'].nunique()} provinces: "
      f"{', '.join(sorted(kept['adm2_en'].unique()))}")
table

**Pause and check (domain knowledge).** The expected neighbors sit in **Bulacan** to the north, **Rizal** to the east, **Cavite** to the southwest, and **Laguna** to the south. Two more checks are worth making on any real file: is a unit listed only because it faces NCR *across Laguna de Bay or Manila Bay*, where some files draw municipal waters into the polygon, and is a true neighbor *missing* because of a gap in the data. The 🎛️ cell near the end re-runs this query at different tolerances to show how sensitive the answer is.

## ✏️ Exercise - count neighbors per province

Using the `kept` neighbors from the query above, build `by_province`: a pandas Series giving the number of adjacent units in each province (`adm2_en`), sorted from most to fewest.

**Hint:** `kept.groupby("adm2_en").size()` counts rows per province; sort the result with `.sort_values(ascending=False)`.

In [ ]:
### START CODE HERE ###   (about 1 line)
by_province = None    # neighbor count per province (adm2_en), most first
### END CODE HERE ###
print(by_province)

## ✅ Check the work

In [ ]:
try:
    ref = kept.groupby("adm2_en").size().sort_values(ascending=False)
    assert by_province is not None, "by_province is None - fill in the code."
    assert isinstance(by_province, pd.Series), f"expected a pandas Series, got {type(by_province)}"
    assert set(by_province.index) == set(ref.index), "some provinces are missing"
    assert np.allclose(by_province.sort_index().values, ref.sort_index().values), \
        "counts do not match - count rows per adm2_en"
    assert int(by_province.sum()) == len(kept), "the counts should add up to every kept neighbor"
    print(f"\u2705 Passed! {len(kept)} neighbors across {len(ref)} provinces: {ref.to_dict()}")
except AssertionError as e:
    print("\u274c Not yet:", e)

### ❓ Question

Read the per-province counts above. How many of the adjacent local government units lie in **Rizal**?

In [ ]:
q_rizal = None   # <- your answer (an integer)

In [ ]:
_rizal = int((kept["adm2_en"] == "Rizal").sum())
if q_rizal == _rizal:
    print(f"\u2705 Correct! {_rizal} adjacent units are in Rizal.")
else:
    print(f"\u274c Not yet: expected {_rizal}, got {q_rizal}.")

## A reusable study-area polygon

The last step packages the result for later notebooks: the 17 NCR cities and the kept neighbors in one GeoDataFrame with a `group` label, plus the single dissolved NCR outline used to clip roads and centers. Coordinates are stored back in degrees (4326), the sharing system, while any measurement happens in meters. The neighbor polygons are clipped to the loaded window, so their outer edges follow the bounding box rather than the true border; the NCR outline is complete.

In [ ]:
ncr_cities["group"] = "NCR"
kept["group"] = "Adjacent"
ncr_cities["lgu"] = ncr_cities["adm3_pcode"].map(NCR_NAMES)
kept["lgu"] = kept["adm3_en"]

keep_cols = ["adm3_pcode", "lgu", "adm2_en", "group", "geometry"]
study_area = gpd.GeoDataFrame(
    pd.concat([ncr_cities[keep_cols], kept[keep_cols]], ignore_index=True),
    geometry="geometry", crs=METRIC_CRS,
).to_crs(STORE_CRS)

ncr_outline_4326 = gpd.GeoSeries([ncr_outline], crs=METRIC_CRS).to_crs(STORE_CRS).iloc[0]
print(f"study_area: {len(study_area)} units "
      f"({(study_area['group']=='NCR').sum()} NCR + {(study_area['group']=='Adjacent').sum()} adjacent)")
study_area.drop(columns="geometry").head()

In [ ]:
# A quick static map of the study area: NCR in red, neighbors by province.
fig, ax = plt.subplots(figsize=(8, 9))
study_area[study_area["group"] == "NCR"].plot(
    ax=ax, color="#d1495b", alpha=0.65, edgecolor="white", linewidth=0.7)
prov_colors = dict(zip(sorted(study_area.loc[study_area["group"] == "Adjacent", "adm2_en"].unique()),
                       ["#8fb996", "#e9c46a", "#a3c4f3", "#f4a261", "#cdb4db"]))
for prov, col in prov_colors.items():
    study_area[(study_area["group"] == "Adjacent") & (study_area["adm2_en"] == prov)].plot(
        ax=ax, color=col, alpha=0.7, edgecolor="white", linewidth=0.5)
gpd.GeoSeries([ncr_outline_4326], crs=STORE_CRS).boundary.plot(ax=ax, color="#7a1022", linewidth=1.8)
handles = [plt.Rectangle((0, 0), 1, 1, color="#d1495b", alpha=0.65)] + \
          [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.7) for c in prov_colors.values()]
ax.legend(handles, ["NCR"] + [f"{p} (adjacent)" for p in prov_colors], loc="lower left", fontsize=8)
ax.set_title(f"Study area: NCR ({(study_area['group']=='NCR').sum()} LGUs) "
             f"+ {(study_area['group']=='Adjacent').sum()} adjacent units")
ax.set_axis_off()
plt.show()

## 🎛️ Play around - move the tolerance and the border cut-off

The neighbor list rests on two choices: the buffer that absorbs slivers, and the minimum shared border that separates a real neighbor from a near miss. Shrink `min_border_m` toward zero and watch corner-only units reappear; raise it and watch short-border units drop. Change the two numbers, re-run, and compare the counts.

In [ ]:
def neighbor_count(buffer_m=150, min_border_m=500):
    z = ncr_outline.buffer(buffer_m)
    hit = outside.iloc[outside.sindex.query(z, predicate="intersects")]
    c = outside[outside["adm3_pcode"].isin(hit["adm3_pcode"].unique())]
    d = c.dissolve(by="adm3_pcode", as_index=False, aggfunc="first")
    d["shared_km"] = d.boundary.intersection(z).length / 1000
    keep = d[d["shared_km"] * 1000 >= min_border_m]
    return len(keep), sorted(keep["adm3_en"].tolist())

for mb in (100, 500, 2000):
    n, names = neighbor_count(min_border_m=mb)
    print(f"min_border_m={mb:>5}: {n:>2} neighbors -> {', '.join(names)}")

In [ ]:
# Interactive in Colab / Jupyter / VS Code (static fallback elsewhere).
try:
    from ipywidgets import interact, IntSlider
    interact(lambda min_border_m=500: print(neighbor_count(min_border_m=min_border_m)[0], "neighbors"),
             min_border_m=IntSlider(min=0, max=3000, step=100, value=500))
except Exception:
    pass

## References & further reading

- [GeoPandas, user guide](https://geopandas.org/en/stable/docs/user_guide.html)
- [GeoPandas, managing projections (CRS)](https://geopandas.org/en/stable/docs/user_guide/projections.html)
- [GeoPandas, aggregating with dissolve](https://geopandas.org/en/stable/docs/user_guide/aggregation_with_dissolve.html)
- [Shapely, binary predicates (touches, intersects)](https://shapely.readthedocs.io/en/stable/manual.html#binary-predicates)
- [EPSG:32651, UTM zone 51N](https://epsg.io/32651)
- [Philippine Standard Geographic Code (PSGC)](https://psa.gov.ph/classification/psgc)
- [GeoPackage format](https://www.geopackage.org/)

## Recap

- A GeoPackage holds named **layers**, and each layer is a **table with shapes**. The header reads without loading the polygons, and a bounding box keeps the read small.
- **The coordinate system matters.** Area in square degrees has no fixed ground size, and the schoolbook "111 km per degree" overstates NCR areas by about 4%. Store in degrees (4326), measure in meters (32651), and check `.crs` whenever datasets meet.
- A PSGC p-code carries its own hierarchy, so NCR is every row with `adm1_pcode == "PH13"`, whatever the name says.
- Cities were built from barangays with a **dissolve**, giving the expected 16 cities plus Pateros, 17 units in all.
- Neighbors came from a **spatial adjacency query**, with buffered `intersects` to absorb slivers and a shared-border length to drop near misses: 12 units across Bulacan, Rizal, Cavite, and Laguna.

**For discussion**
1. TESDA records name cities in many ways ("Quezon City", "QC", "Q.C.", "Lungsod Quezon"). What breaks when those are joined to boundaries by name, and why does a PSGC code column fix it?
2. Should the adjacent area be defined by **shared border**, as here, or by **travel time**, every unit within 60 minutes of an NCR center? Which one matches how learners actually move?
3. The neighbor list changed with the buffer and border cut-off. Which setting would be defended in a report, and what evidence would support it?